In [2]:
import numpy as np


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW


In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)


C:\Users\marti\XAI4LLMsMetaToolkit\xai-llm-router\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from alibi.explainers import GradientSimilarity
from sklearn.model_selection import train_test_split
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings("ignore")

ImportError: cannot import name 'GradientSimilarity' from 'alibi.explainers' (C:\Users\marti\XAI4LLMsMetaToolkit\xai-llm-router\.venv\Lib\site-packages\alibi\explainers\__init__.py)

In [6]:
from alibi.explainers.similarity.grad import GradientSimilarity

ModuleNotFoundError: No module named 'alibi.explainers.similarity'

In [7]:
import alibi; print(alibi.__version__)  # needs to be >= 0.7.0


0.5.5


In [ ]:
"""
BERT Fine-tuning + Alibi GradientSimilarity Pipeline
=====================================================
Full pipeline for:
  1. Fine-tuning a BERT-like HuggingFace model on a small binary dataset
  2. Wrapping it so Alibi's GradientSimilarity can use it
  3. Fitting the explainer on the training set
  4. At inference: classifying a new text + finding the most similar training examples

Install:
    pip install transformers torch alibi

Notes on small datasets (30 pos / 30 neg):
  - Use precompute_grads=False (default): with 60 examples this is fast enough on-the-fly
  - Use sim_fn='grad_cos': more stable than dot product on small sets
  - X_train passed to alibi must be tokenized tensors, not raw strings (see below)
"""




# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

MODEL_NAME = "distilbert-base-uncased"   # any BERT-like HuggingFace model
MAX_LEN    = 128
BATCH_SIZE = 8
EPOCHS     = 5
LR         = 2e-5
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"



class TextDataset(Dataset):
    def __init__(self, encodings: Dict, labels: List[int]):
        self.encodings = encodings   # dict with input_ids, attention_mask (already tensors)
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }


def tokenize(texts: List[str], tokenizer, max_len: int = MAX_LEN) -> Dict:
    return tokenizer(
        texts,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )


# ─────────────────────────────────────────────
# 2. FINE-TUNING
# ─────────────────────────────────────────────

def fine_tune(
    train_texts: List[str],
    train_labels: List[int],
    model_name: str = MODEL_NAME,
    epochs: int = EPOCHS,
    lr: float = LR,
    max_len: int = MAX_LEN,
    val_split: float = 0.15,
) -> Tuple[AutoModelForSequenceClassification, AutoTokenizer]:

    print(f"Fine-tuning {model_name} on {len(train_texts)} examples  [device={DEVICE}]\n")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(DEVICE)

    # Split
    tr_texts, val_texts, tr_labels, val_labels = train_test_split(
        train_texts, train_labels,
        test_size=val_split,
        stratify=train_labels,
        random_state=42,
    )

    tr_enc   = tokenize(tr_texts,  tokenizer, max_len)
    val_enc  = tokenize(val_texts, tokenizer, max_len)
    tr_data  = TextDataset(tr_enc,  tr_labels)
    val_data = TextDataset(val_enc, val_labels)
    tr_loader  = DataLoader(tr_data,  batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(tr_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps,
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tr_loader:
            optimizer.zero_grad()
            out  = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
                labels=batch["label"].to(DEVICE),
            )
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += out.loss.item()

        # Validation
        model.eval()
        val_loss, correct = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                out = model(
                    input_ids=batch["input_ids"].to(DEVICE),
                    attention_mask=batch["attention_mask"].to(DEVICE),
                    labels=batch["label"].to(DEVICE),
                )
                val_loss += out.loss.item()
                preds     = out.logits.argmax(dim=-1)
                correct  += (preds == batch["label"].to(DEVICE)).sum().item()

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"train_loss={total_loss/len(tr_loader):.4f} | "
            f"val_loss={val_loss/len(val_loader):.4f} | "
            f"val_acc={correct/len(val_texts):.2f}"
        )

    print("\nFine-tuning complete.\n")
    return model, tokenizer


# ─────────────────────────────────────────────
# 3. ALIBI WRAPPER
#
# Alibi's GradientSimilarity needs a callable that:
#   - takes a batch of inputs (here: dict with input_ids + attention_mask)
#   - returns raw logits as a torch.Tensor
#
# The trick for HuggingFace models: we wrap them in a thin nn.Module
# that accepts a dict and returns logits, which Alibi can differentiate through.
# ─────────────────────────────────────────────

class AlibiCompatibleBERT(nn.Module):
    """
    Thin wrapper so Alibi can call the model with a single tensor input.

    Alibi passes X (the training data) directly to the model during gradient
    computation. Since we stored X as a stacked tensor of shape
    (n_samples, 2, max_len) — where dim 1 holds [input_ids, attention_mask] —
    this wrapper unpacks it before forwarding.
    """
    def __init__(self, hf_model: AutoModelForSequenceClassification):
        super().__init__()
        self.model = hf_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, 2, max_len)
        input_ids      = x[:, 0, :].long()
        attention_mask = x[:, 1, :].long()
        out = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return out.logits


# ─────────────────────────────────────────────
# 4. PREPARE TRAINING DATA FOR ALIBI
#
# Alibi's fit() expects X_train as a numpy array or list.
# We encode all training texts and pack them as (n, 2, max_len) tensors,
# then convert to numpy so Alibi can store them and replay them for gradient computation.
# ─────────────────────────────────────────────

def prepare_alibi_data(
    texts: List[str],
    labels: List[int],
    tokenizer,
    max_len: int = MAX_LEN,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns:
        X: np.ndarray of shape (n, 2, max_len)  — stacked [input_ids, attention_mask]
        Y: np.ndarray of shape (n,)              — integer labels
    """
    enc = tokenize(texts, tokenizer, max_len)
    # Stack input_ids and attention_mask along a new dim → (n, 2, max_len)
    X = torch.stack([enc["input_ids"], enc["attention_mask"]], dim=1).numpy()
    Y = np.array(labels)
    return X, Y


# ─────────────────────────────────────────────
# 5. INFERENCE + SIMILARITY
# ─────────────────────────────────────────────

def predict_with_similar(
    text: str,
    wrapped_model: AlibiCompatibleBERT,
    tokenizer,
    explainer: GradientSimilarity,
    train_texts: List[str],
    train_labels: List[int],
    max_len: int = MAX_LEN,
    top_k: int = 5,
) -> Dict:
    """
    1. Classifies the input text
    2. Uses Alibi's GradientSimilarity to find the most similar training examples
    """
    wrapped_model.eval()

    # Tokenize test input → (1, 2, max_len)
    enc   = tokenize([text], tokenizer, max_len)
    X_test = torch.stack([enc["input_ids"], enc["attention_mask"]], dim=1)

    # Classification
    with torch.no_grad():
        logits = wrapped_model(X_test.to(DEVICE))
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_label = int(np.argmax(probs))

    # Alibi explanation — pass numpy array (n=1, 2, max_len)
    X_test_np = X_test.numpy()
    explanation = explainer.explain(X_test_np)

    # Alibi returns most_similar as raw X arrays; map back to text using ordered_indices
    ordered_idx = explanation.data["ordered_indices"][0]   # (n_train,) sorted by similarity
    scores      = explanation.data["scores"][0]            # (n_train,) similarity scores

    top_results = []
    for rank, idx in enumerate(ordered_idx[:top_k], start=1):
        top_results.append({
            "rank":       rank,
            "text":       train_texts[idx],
            "label":      train_labels[idx],
            "label_name": "positive" if train_labels[idx] == 1 else "negative",
            "score":      float(scores[idx]),
        })

    return {
        "text":             text,
        "predicted_label":  pred_label,
        "label_name":       "positive" if pred_label == 1 else "negative",
        "confidence":       float(probs[pred_label]),
        "probabilities":    {"negative": float(probs[0]), "positive": float(probs[1])},
        "similar_examples": top_results,
    }


def print_result(result: Dict):
    print("=" * 65)
    print(f"INPUT      : {result['text']}")
    print(f"PREDICTION : {result['label_name'].upper()} ({result['confidence']:.2%})")
    print(f"PROBS      : neg={result['probabilities']['negative']:.3f}  pos={result['probabilities']['positive']:.3f}")
    print(f"\nTOP-{len(result['similar_examples'])} MOST SIMILAR TRAINING EXAMPLES (Alibi GradientSimilarity):")
    for ex in result["similar_examples"]:
        tag = "✅ POS" if ex["label"] == 1 else "❌ NEG"
        print(f"  [{ex['rank']}] score={ex['score']:.4f} | {tag} | {ex['text'][:75]}")
    print("=" * 65 + "\n")





C:\Users\marti\XAI4LLMsMetaToolkit\xai-llm-router\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# ── Replace with your actual data ─────────────────────────────
positive_texts = [
    "This product is absolutely amazing and works perfectly.",
    "I love this, it exceeded all my expectations.",
    "Fantastic quality and fast delivery, very happy.",
    "Best purchase I've made this year, highly recommend.",
    "Great value for money, works exactly as described.",
] * 6
positive_texts = positive_texts[:30]

negative_texts = [
    "Terrible product, broke after one day.",
    "Complete waste of money, very disappointed.",
    "Does not work as advertised, poor quality.",
    "Would not recommend, cheaply made and useless.",
    "Worst purchase ever, returning immediately.",
] * 6
negative_texts = negative_texts[:30]
# ──────────────────────────────────────────────────────────────

all_texts  = positive_texts + negative_texts
all_labels = [1] * 30 + [0] * 30

# Shuffle
rng     = np.random.default_rng(42)
indices = rng.permutation(len(all_texts))
all_texts  = [all_texts[i]  for i in indices]
all_labels = [all_labels[i] for i in indices]

# ── Step 1: Fine-tune ──────────────────────────────────────────
hf_model, tokenizer = fine_tune(all_texts, all_labels)

# ── Step 2: Wrap model for Alibi ──────────────────────────────
wrapped_model = AlibiCompatibleBERT(hf_model).to(DEVICE)
wrapped_model.eval()

# ── Step 3: Prepare training data for Alibi ───────────────────
X_train, Y_train = prepare_alibi_data(all_texts, all_labels, tokenizer)

# ── Step 4: Init + fit Alibi GradientSimilarity ───────────────
# sim_fn options: 'grad_cos' (recommended), 'grad_dot', 'grad_asym_dot'
# precompute_grads=True is faster at explain() time but uses more memory at fit() time.
# With 60 examples, either setting is fast enough.
print("Fitting Alibi GradientSimilarity explainer...")
explainer = GradientSimilarity(
    predictor=wrapped_model,
    loss_fn=nn.CrossEntropyLoss(),
    sim_fn="grad_cos",
    task="classification",
    precompute_grads=False,   # set True to speed up explain() at the cost of fit() time
    backend="pytorch",
    device=DEVICE,
)
explainer.fit(X_train, Y_train)
print("Explainer ready.\n")

# ── Step 5: Predict + explain ─────────────────────────────────
test_texts = [
    "This item is really good and I would buy it again.",
    "Absolute garbage, nothing works and support is useless.",
]

for text in test_texts:
    result = predict_with_similar(
        text,
        wrapped_model,
        tokenizer,
        explainer,
        all_texts,
        all_labels,
        top_k=5,
    )
    print_result(result)

# ── Optional: save model + explainer ─────────────────────────
# hf_model.save_pretrained("./my_bert_model")
# tokenizer.save_pretrained("./my_bert_model")
# import dill; dill.dump(explainer, open("explainer.pkl", "wb"))